In [ ]:
from dotenv import dotenv_values
import gradio as gr
from transformers import pipeline
import numpy as np
import re
import requests

ENV_ACCESS=dotenv_values()
API_URL = ENV_ACCESS['API_URL']


p_trascriber = pipeline("automatic-speech-recognition", model="openai/whisper-base.en")

def call_generate():
    global g_str_splitted_text
    payload = {"prompt": "tell me a joke about cats"}
        
    response = requests.post(
        f"{API_URL}/generate",
        json=payload
    )
    g_str_splitted_text = response.text if response.ok else "Error in response"
    print("g_str_splitted_text=",g_str_splitted_text)
    return g_str_splitted_text


# where the tokenized string from text transciption is
def transcribe(audio):
    sr, y = audio
    y = y.astype(np.float32)
    if len(y) > 0:
        y /= np.max(np.abs(y))
    # my text from voice 
    text_from_voice = str(p_trascriber({"sampling_rate": sr, "raw": y})["text"])
    #global g_splitted_text_from_Speech 
    global g_list_splitted_text
    g_list_splitted_text= re.split('\. | \,', text_from_voice)
    print("g_splitted_text_from_Speech=", g_list_splitted_text)

    print("My_voice_transcribed=", p_trascriber({"sampling_rate": sr, "raw": y})["text"])
    #return transcriber({"sampling_rate": sr, "raw": y})["text"]
    return text_from_voice

# audio interface in Gradio for recording 
audio_interf = gr.Interface(
    transcribe,
    gr.Audio(source="microphone", label="Joke about?", show_api="True",),
    "text",
    css="footer {display:none !important}",
)
title_textbox = gr.Textbox(value=" Yassine Maalej ",label ="TEST",css="footer {display:none !important}",)
# gradio textbox from speech to joke
output_textbox = gr.Textbox(value="Speak about a joke", label="Mini Jokes", live=True,css="footer {display:none !important}",)
with gr.Blocks() as demo:
    # Render Title, audio record adn transcripts and genrate button and text fields
    title_textbox.render()
    # Audio interface for real-time render
    audio_interf.render()

    # generate a joke from transcribed voice to text prompt
    btn_generate = gr.Button("Generate Joke")

    
    btn_generate.click(call_generate, inputs=None , outputs=output_textbox)

    # box with response joke 
    output_textbox.render()


demo.launch()